In [48]:
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal
from typing import List

In [49]:
@dataclass
class Operation:
    name: str
    amount: Decimal
    date: datetime
    balance_after: Decimal
    status: str
    used_credit: bool = False

In [50]:
class Account:
    def __init__(self, account_holder: str, balance=0):
        balance = self._to_decimal(balance)
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder
        self._balance = Account._to_decimal(balance)
        self.operations_history: List[Operation] = []

    @staticmethod
    def _to_decimal(value) -> Decimal:
        # лучше через str
        return Decimal(str(value))

    def deposit(self, amount) -> bool:
        amount = self._to_decimal(amount)
        if amount <= 0:
            self._log("deposit", amount, "fail")
            return False

        self._balance += amount # normalized by Decimal(str())
        self._log("deposit", amount, "success")
        return True

    def withdraw(self, amount) -> bool:
        """Неудачная попытка сохраняется в истории"""
        amount = self._to_decimal(amount)
        if amount <= 0 or amount > self._balance:
            self._log("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._log("withdraw", amount, "success")
        return True

    def get_balance(self) -> Decimal:
        return self._balance

    def get_history(self) -> List[dict]:
        """Возвращает историю операций"""
        return [self._operation_to_dict(op) for op in self.operations_history]

    def _operation_to_dict(self, op: Operation) -> dict:
        return {
            "operation": op.name,
            "amount": op.amount,
            "date": op.date.strftime("%d-%m-%Y %H:%M:%S"),
            "balance_after": op.balance_after,
            "status": op.status,
        }

    def _log(self, name: str, amount: Decimal, status: str,
             used_credit: bool = False):
        self.operations_history.append(
            Operation(name, amount, datetime.now(), self._balance, status,
                      used_credit=used_credit)
        )

In [51]:
class CreditAccount(Account):
    """аккаунт, баланс которого может уйти в минус (не ниже credit_limit)"""

    def __init__(self, account_holder: str, balance=0, credit_limit=0):
        balance = self._to_decimal(balance)
        credit_limit = self._to_decimal(credit_limit)
        
        if credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")
        if balance < -credit_limit:
            raise ValueError("Начальный баланс ниже кредитного лимита")

        # базовый конструктор не примет отрицательный баланс,
        # поэтому создаём счёт с нулём и выставляем баланс вручнуб
        super().__init__(account_holder)
        self._balance = balance
        self.credit_limit = credit_limit

    def get_available_credit(self) -> Decimal:
        """Сколько ещё можно потратить"""
        return self._balance + self.credit_limit

    def withdraw(self, amount) -> bool:
        """Снимает деньги в пределах лимита"""
        amount = self._to_decimal(amount)
        if amount <= 0 or self._balance - amount < -self.credit_limit:
            self._log("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._log("withdraw", amount, "success", used_credit=self._balance < 0)
        return True

    def _operation_to_dict(self, op: Operation) -> dict:
        record = super()._operation_to_dict(op)
        record["used_credit"] = op.used_credit
        return record

In [52]:
acc = CreditAccount("Ivan", 0, 450)
print(acc.holder)
print(acc.get_balance())
acc.get_available_credit()

Ivan
0


Decimal('450')

In [53]:
acc.deposit(50)
print("balance:", acc.get_balance())
print("available credit:", acc.get_available_credit())

balance: 50
available credit: 500


In [54]:
acc.withdraw(30)  # хватает

True

In [55]:
acc.get_history()[-1]  # последняя операция

{'operation': 'withdraw',
 'amount': Decimal('30'),
 'date': '08-11-2026 00:56:14',
 'balance_after': Decimal('20'),
 'status': 'success',
 'used_credit': False}

In [56]:
success = acc.withdraw(470)
success

True

In [57]:
acc.get_history()[-1]  # used_credit=True

{'operation': 'withdraw',
 'amount': Decimal('470'),
 'date': '08-11-2026 00:56:23',
 'balance_after': Decimal('-450'),
 'status': 'success',
 'used_credit': True}

In [58]:
success = acc.withdraw(30)
success

False

In [59]:
acc.get_history()[-1]  # лимит исчерпан, fail

{'operation': 'withdraw',
 'amount': Decimal('30'),
 'date': '08-11-2026 00:56:33',
 'balance_after': Decimal('-450'),
 'status': 'fail',
 'used_credit': False}

In [60]:
print("balance:", acc.get_balance())
print("available credit:", acc.get_available_credit())

balance: -450
available credit: 0


In [61]:
for op in acc.get_history():
    print(op["operation"], op["status"], op["used_credit"])

deposit success False
withdraw success False
withdraw success True
withdraw fail False
